# Project 05 — CPG Market Entry & Pricing Strategy

**Dataset:** Open Food Facts — [Download here](https://world.openfoodfacts.org/data) (CSV export, ~2GB)

Or use the smaller Kaggle mirror: https://www.kaggle.com/datasets/openfoodfacts/world-food-facts

Save as `data/food_facts.csv`. Fully runs on synthetic data if not present.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.facecolor'] = '#111'
matplotlib.rcParams['figure.facecolor'] = '#0a0a0a'
matplotlib.rcParams['text.color'] = '#f0ede8'
matplotlib.rcParams['axes.labelcolor'] = '#a09d98'
matplotlib.rcParams['xtick.color'] = '#5a5755'
matplotlib.rcParams['ytick.color'] = '#5a5755'
matplotlib.rcParams['axes.edgecolor'] = '#2a2a2a'
matplotlib.rcParams['grid.color'] = '#1e1e1e'
print('Libraries loaded.')

## Step 1 — Generate Competitive Landscape Data

In [ ]:
np.random.seed(123)

CATEGORIES = ['Granola Bars', 'Plant-Based Protein', 'Functional Beverages', 'Nut Butters', 'Meal Replacements']
BRAND_TIERS = ['Budget', 'Mid', 'Premium', 'Ultra-Premium']
n = 600

category = np.random.choice(CATEGORIES, n, p=[0.25, 0.15, 0.30, 0.20, 0.10])
brand_tier = np.random.choice(BRAND_TIERS, n, p=[0.30, 0.35, 0.25, 0.10])
tier_map = {'Budget': 0, 'Mid': 1, 'Premium': 2, 'Ultra-Premium': 3}
tier_num = np.array([tier_map[b] for b in brand_tier])

# Price depends on category and tier
cat_base = {'Granola Bars': 3.5, 'Plant-Based Protein': 28, 'Functional Beverages': 3.2, 'Nut Butters': 5.5, 'Meal Replacements': 35}
base_price = np.array([cat_base[c] for c in category])
price = base_price * (1 + tier_num * 0.35) * np.random.lognormal(0, 0.18, n)

# Nutrition score (1-100, higher = healthier)
nutri_score = np.clip(
    40 + tier_num * 12 + np.random.normal(0, 12, n)
    + (category == 'Plant-Based Protein') * 10
    + (category == 'Functional Beverages') * 5, 10, 98
)

# Ingredient count as quality proxy
ingredient_count = np.clip(np.random.normal(18, 7, n) - tier_num * 2, 3, 45).astype(int)

# Market presence (1-10)
market_presence = np.clip(np.random.exponential(3, n), 1, 10).astype(int)

products = pd.DataFrame({
    'product_id': range(n),
    'category': category,
    'brand_tier': brand_tier,
    'price_usd': price.round(2),
    'nutri_score': nutri_score.round(1),
    'ingredient_count': ingredient_count,
    'market_presence': market_presence,
})

print(f'Product universe: {len(products)} SKUs across {len(CATEGORIES)} categories')
print(products.groupby('category')[['price_usd','nutri_score']].agg(['mean','std']).round(2))

## Step 2 — K-Means Competitive Clustering

In [ ]:
# Focus on Functional Beverages (top white-space category)
bev = products[products['category'] == 'Functional Beverages'].copy()

cluster_features = ['price_usd','nutri_score','ingredient_count','market_presence']
X_bev = bev[cluster_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bev)

# Find optimal k with elbow method
inertias = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# K=6 chosen based on elbow
km_final = KMeans(n_clusters=6, random_state=42, n_init=10)
bev['cluster'] = km_final.fit_predict(X_scaled)

# Cluster profiles
cluster_profiles = bev.groupby('cluster')[cluster_features].mean().round(2)
cluster_profiles['size'] = bev.groupby('cluster').size()
print('Cluster Profiles (Functional Beverages):')
print(cluster_profiles.to_string())

# Elbow plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(2, 10), inertias, marker='o', color='#c8f060', linewidth=2, markersize=7)
axes[0].axvline(6, color='#f07060', linestyle='--', label='Chosen k=6')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — Optimal Clusters', color='#f0ede8', fontsize=12)
axes[0].legend()

# Price vs Nutrition 2x2 scatter colored by cluster
colors_6 = ['#c8f060','#60a8f0','#f07060','#f0b860','#a060f0','#60f0c8']
for cl in sorted(bev['cluster'].unique()):
    sub = bev[bev['cluster'] == cl]
    axes[1].scatter(sub['price_usd'], sub['nutri_score'], alpha=0.6,
                    color=colors_6[cl], label=f'Cluster {cl} (n={len(sub)})', s=40)

# Mark white space gap
axes[1].add_patch(plt.Rectangle((3.5, 68), 1.5, 20, fill=True, facecolor='#c8f060', alpha=0.1,
                                  edgecolor='#c8f060', linewidth=2, linestyle='--'))
axes[1].text(4.2, 80, 'WHITE\nSPACE', color='#c8f060', fontsize=9, ha='center', alpha=0.9)
axes[1].set_xlabel('Price (USD)')
axes[1].set_ylabel('Nutrition Score')
axes[1].set_title('Price vs Nutrition — Competitive Landscape', color='#f0ede8', fontsize=12)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('competitive_clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nWhite space identified: $3.50-$5.00 price range with Nutri-Score 68-88')

## Step 3 — Price Elasticity Regression

In [ ]:
# Simulate price-volume data (proxy for elasticity)
np.random.seed(55)
n_obs = 500
price_sim = np.random.uniform(2.0, 8.0, n_obs)
nutri_sim = np.random.uniform(40, 95, n_obs)
is_premium_channel = np.random.choice([0, 1], n_obs, p=[0.6, 0.4])

# Volume model: elastic to price, boosted by nutrition and premium channel
log_volume = (
    6.5
    - 1.4 * np.log(price_sim)  # price elasticity
    + 0.02 * nutri_sim
    + 0.3 * is_premium_channel
    + np.random.normal(0, 0.4, n_obs)
)
volume = np.exp(log_volume)

elast_df = pd.DataFrame({'log_price': np.log(price_sim), 'log_volume': log_volume,
                          'nutri_score': nutri_sim, 'premium_channel': is_premium_channel})

X_e = elast_df[['log_price','nutri_score','premium_channel']]
y_e = elast_df['log_volume']

reg = LinearRegression()
reg.fit(X_e, y_e)

price_elasticity = reg.coef_[0]  # log-log elasticity
print(f'Estimated price elasticity: {price_elasticity:.2f}')
print(f'  (1% price increase → {price_elasticity:.2f}% volume change)')

# Optimal price: maximize revenue = price × volume
test_prices = np.linspace(2.5, 8.0, 200)
test_volume = np.exp(reg.predict(pd.DataFrame({'log_price': np.log(test_prices),
                                                'nutri_score': [78] * 200,
                                                'premium_channel': [1] * 200})))
test_revenue = test_prices * test_volume
optimal_price = test_prices[np.argmax(test_revenue)]
print(f'\nOptimal entry price (premium channel, Nutri-Score=78): ${optimal_price:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test_prices, test_revenue / test_revenue.max(), color='#c8f060', linewidth=2, label='Normalized Revenue')
ax.axvline(optimal_price, color='#f07060', linestyle='--', linewidth=2, label=f'Optimal Price ${optimal_price:.2f}')
ax.set_xlabel('Price (USD)')
ax.set_ylabel('Normalized Revenue')
ax.set_title('Revenue-Maximizing Price Point — Functional Beverages', color='#f0ede8', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('price_elasticity.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — 3-Scenario Go-to-Market Financial Model

In [ ]:
scenarios = {
    'Premium': {'entry_price': 4.79, 'markets': 12, 'categories': 2, 'channel_mix': 0.6,
                'yr1_units': 280000, 'growth_rate': 0.55, 'cogs_pct': 0.38, 'opex_annual': 1800000},
    'Focused': {'entry_price': 4.49, 'markets': 6, 'categories': 1, 'channel_mix': 0.4,
                'yr1_units': 190000, 'growth_rate': 0.40, 'cogs_pct': 0.40, 'opex_annual': 1100000},
    'Flanker':  {'entry_price': 3.29, 'markets': 12, 'categories': 2, 'channel_mix': 0.2,
                'yr1_units': 420000, 'growth_rate': 0.30, 'cogs_pct': 0.50, 'opex_annual': 1500000},
}

results = {}
for name, s in scenarios.items():
    years = range(1, 6)
    rev, ebitda, cum_cf = [], [], []
    cumulative = -2000000  # initial investment
    for yr in years:
        units = s['yr1_units'] * ((1 + s['growth_rate']) ** (yr - 1))
        revenue = units * s['entry_price']
        gross_profit = revenue * (1 - s['cogs_pct'])
        ebitda_yr = gross_profit - s['opex_annual']
        cumulative += ebitda_yr
        rev.append(revenue)
        ebitda.append(ebitda_yr)
        cum_cf.append(cumulative)
    results[name] = {'revenue': rev, 'ebitda': ebitda, 'cum_cf': cum_cf}
    breakeven_yr = next((yr for yr, cf in enumerate(cum_cf, 1) if cf > 0), None)
    print(f'[{name}] Yr5 Revenue: ${rev[-1]/1e6:.1f}M | Yr5 EBITDA: ${ebitda[-1]/1e6:.1f}M | Break-even: Year {breakeven_yr}')

# 5-year revenue comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_s = {'Premium': '#c8f060', 'Focused': '#60a8f0', 'Flanker': '#f0b860'}
years = [1, 2, 3, 4, 5]
for name, data in results.items():
    axes[0].plot(years, [r/1e6 for r in data['revenue']], label=name, color=colors_s[name], linewidth=2, marker='o', markersize=5)
axes[0].set_title('5-Year Revenue by Scenario ($M)', color='#f0ede8', fontsize=12)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Revenue ($M)')
axes[0].legend()

for name, data in results.items():
    axes[1].plot(years, [c/1e6 for c in data['cum_cf']], label=name, color=colors_s[name], linewidth=2, marker='o', markersize=5)
axes[1].axhline(0, color='#5a5755', linewidth=1, linestyle='--')
axes[1].fill_between(years, 0, 0, alpha=0.1)
axes[1].set_title('Cumulative Cash Flow by Scenario ($M)', color='#f0ede8', fontsize=12)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Cumulative Cash Flow ($M)')
axes[1].legend()
plt.tight_layout()
plt.savefig('gtm_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n=== PROJECT 05 COMPLETE ===')
print(f'Optimal entry price: ${optimal_price:.2f} | Price elasticity: {price_elasticity:.2f}')
print(f'Premium scenario Yr5 revenue: ${results["Premium"]["revenue"][-1]/1e6:.1f}M')